# Part 1 — Notebook 01: MadGraph Process Setup and LHE Generation

## Pedagogical Goal & Overview

Welcome to **Part 1 — Notebook 01** of the experimental High-Energy Physics (HEP) Monte Carlo training series.

In this notebook, you will execute the first hands-on step of event generation:
$$\text{Process Definition} \rightarrow \text{MadGraph Process/Run Cards} \rightarrow \text{Parton-Level Events (LHE)} \rightarrow \text{Event Record Inspection}$$

### Primary Pedagogical Process
We study proton-proton production of a $Z$ boson recoiling against a hard matrix-element jet, with the $Z$ boson forced to decay to a bottom-quark pair:
```text
p p > z j, z > b b~
```

### Full Monte Carlo Event Pipeline Context
It is critical to understand where this notebook fits in the experimental physics chain:
$$\underbrace{\text{Hard Scattering (ME)} \rightarrow \text{Resonance Decay}}_{\text{Notebook 01 (MadGraph LHE)}} \rightarrow \text{Parton Shower} \rightarrow \text{Hadronization} \rightarrow \text{Detector Sim} \rightarrow \text{Reconstruction} \rightarrow \text{Analysis}$$

In this notebook, we operate purely at the **parton-level matrix element (ME)** stage.

## Step 1: Environment Setup & Google Drive Workspace Initialization

Before executing event generators, we verify our computing environment in Google Colab:
1. Verify compiler prerequisites (`gfortran`).
2. Mount **Google Drive** so all generated cards, test runs, production LHE event files, and Feynman diagram PDFs are saved directly to persistent Google Drive storage (`/content/drive/MyDrive/MadGraph_Zbb_Outputs`).

In [ ]:
import os
import sys
import subprocess

print("=== Environment Verification ===")
print(f"Python version: {sys.version.split()[0]}")

# Verify gfortran compiler
try:
    gfortran_info = subprocess.check_output(["gfortran", "--version"]).decode('utf-8').split('\n')[0]
    print(f"GFortran compiler: {gfortran_info}")
except Exception as e:
    print(f"WARNING: gfortran check failed ({e}). MadGraph requires gfortran to compile matrix elements.")

# Set up persistent output directory in Google Drive (or local working directory fallback)
if os.path.exists("/content"):
    try:
        from google.colab import drive
        print("\nMounting Google Drive for persistent file storage...")
        drive.mount('/content/drive', force_remount=False)
        output_dir = "/content/drive/MyDrive/MadGraph_Zbb_Outputs"
    except Exception as e:
        print(f"Notice: Drive mount skipped ({e}). Using local workspace.")
        output_dir = os.getcwd()
else:
    output_dir = os.getcwd()

os.makedirs(output_dir, exist_ok=True)
mg5_dir = "/content/MG5_aMC" if os.path.exists("/content") else os.path.join(os.getcwd(), "MG5_aMC")

print(f"\nMadGraph installation target: {mg5_dir}")
print(f"Persistent output workspace : {output_dir}")


## Step 2: Download and Install MadGraph5_aMC@NLO

We download the official MadGraph5_aMC@NLO release archive (`MG5_aMC_v3.5.16.tar.gz`), extract it to `/content/MG5_aMC`, and confirm that the executable `./bin/mg5_aMC` exists.

In [ ]:
mg5_tar = "MG5_aMC_v3.5.16.tar.gz"
mg5_url = "https://launchpad.net/mg5amcnlo/3.0/3.7.x/+download/MG5_aMC_v3.5.16.tar.gz"

if not os.path.exists(mg5_dir):
    print("Downloading MadGraph5_aMC@NLO v3.5.16...")
    !wget -q {mg5_url} -O {mg5_tar}
    print("Extracting archive...")
    !tar -xzf {mg5_tar}
    # Corrected the directory name from dots to underscores
    if os.path.exists("MG5_aMC_v3_5_16") and not os.path.exists(mg5_dir):
        os.rename("MG5_aMC_v3_5_16", mg5_dir)
    print(f"MadGraph5 installed at {mg5_dir}")
else:
    print(f"MadGraph5 already installed at {mg5_dir}")

mg5_exe = os.path.join(mg5_dir, "bin", "mg5_aMC")
assert os.path.exists(mg5_exe), f"ERROR: MadGraph executable not found at {mg5_exe}"
print(f"SUCCESS: Verified MadGraph executable at {mg5_exe}")


## Step 3: Integrated Physics Foundations — Standard Model, Boosted Z, & Event Pipeline

### 1. Minimal Standard Model Orientation
- **Quarks**: $u, d, c, s, t, b$. 
- **Gauge Bosons**: Photon ($\gamma$), $W^\pm$, $Z^0$, gluon ($g$). The $Z$ boson is a neutral heavy electroweak gauge boson ($m_Z \approx 91.2\text{ GeV}$).

### 2. $Z$ Production and Hadronic Decay
At the LHC ($\sqrt{s} = 13\text{ TeV}$), $Z$ bosons are produced via quark-antiquark annihilation ($q\bar{q} \to Z$). $Z \to b\bar{b}$ has a large branching ratio (~15.1%), but is dominated by background from pure Quantum Chromodynamics (QCD) multijet events ($pp \to jj$).

### 3. Boosted Topology
By requiring a $Z$ boson with high transverse momentum ($p_T^Z > 150\text{ GeV}$), relativistic boost collimates the decay partons ($b, \bar{b}$):
$$\Delta R_{b\bar{b}} \approx \frac{2m_Z}{p_T^Z}$$

### 4. Generator Truth vs. Reconstructed Data
- **Generator Truth**: The exact, simulated four-momenta of matrix-element partons ($PID = \pm 5$). Unavailable in real recorded collider data!
- **Reconstructed Jet**: An algorithmic proxy created from calorimeter energy deposits or charged tracks, NOT identical to an initial parton.
- **LHE Record**: Stores matrix-element truth objects ($b, \bar{b}, Z$), NOT detector-reconstructed jets.

---

> [!IMPORTANT]
> ### Exercise 3: Physics Pipeline & Truth Identification
> Answer the following questions based on the pipeline above:
> 1. Which stages of the event pipeline ($\text{ME} \rightarrow \text{Decay} \rightarrow \text{Shower} \rightarrow \text{Hadronization} \rightarrow \text{Detector} \rightarrow \text{Reconstruction} \rightarrow \text{Analysis}$) are performed by MadGraph in this notebook?
> 2. Why is a $b$ quark in an LHE file considered "generator truth" rather than a reconstructed $b$-jet?

<details>
<summary>Click to show Exercise 3 Reference Solution</summary>

<p>1. <b>Pipeline Stages Present</b>: MadGraph performs <b>Hard Scattering (Matrix Element calculation)</b> and <b>Resonance Decay</b> ($Z \to b\bar{b}$). Parton showering, hadronization, detector simulation, and jet reconstruction belong to downstream analysis steps.</p>
<p>2. <b>Generator Truth Partons</b>: Status 1 bottom quarks ($PID = \pm 5, status = 1$) in LHE records are bare matrix-element partons produced directly from $Z$ decay. They have not undergone gluon radiation, hadronization, or calorimeter clustering.</p>

</details>

## Step 4: Create & Review MadGraph Cards

MadGraph uses two plain-text configuration files ("cards") to fully specify event generation.
In Colab we create them directly from Python so you can inspect and modify every parameter.

- **Process Card** (`cards/zbbj_proc_card.dat`): Defines *what* to generate — the physics process, decay chain, and output directory name.
- **Run Card** (`cards/zbbj_run_card.dat`): Defines *how* to generate — beam energies, event count, random seed, and all kinematic selection cuts.

Run the cells below to write the cards to disk, then read and inspect their contents.

In [ ]:
import os

# ── Create the cards directory inside output_dir ───────────────────────────
cards_dir = os.path.join(output_dir, "cards")
os.makedirs(cards_dir, exist_ok=True)

# ── Process Card ──────────────────────────────────────────────────────────────
# Each line is a MadGraph5 command executed in sequence:
#   import model sm   : load the Standard Model particle content & Feynman rules
#   generate ...      : specify the hard-scattering process
#                       "p p > z j"  — a Z boson produced with a hard jet
#                       ", z > b b~" — forced decay of the Z to a b-bbar pair
#   output Zbbj_LO    : write the compiled matrix-element code to this directory
#   launch Zbbj_LO    : run the generation using the settings below

proc_card_lines = [
    "import model sm\n",
    "generate p p > z j, z > b b~\n",
    "output Zbbj_LO\n",
    "launch Zbbj_LO\n",
]
proc_card_content = ''.join(proc_card_lines)

proc_card_path = os.path.join(cards_dir, "zbbj_proc_card.dat")
with open(proc_card_path, "w") as f:
    f.write(proc_card_content)

# ──── Print the card so you can inspect (and edit the list above if needed) ───
print(f"=== Process Card written to: {proc_card_path} ===\n")
print(proc_card_content)


In [ ]:
# ── Run Card ────────────────────────────────────────────────────────────────
# This card controls all numerical parameters of the generation run.

# ┌─────────────────────────────────────────────────────────────────────────┐
# │  Edit these values to customise your run before writing the card.       │
# │  nevents is set to -1 here — the actual count is passed at launch time  │
# │  via 'set nevents N' so you can change it without rewriting the card.   │
# └─────────────────────────────────────────────────────────────────────────┘
nevents = -1     # -1 = controlled at launch time via 'set nevents N'
iseed   = 0      # Random seed  (0 = auto-generate a random seed)
ptj_cut = 150.0  # Minimum pT of the recoiling jet [GeV]
ptZ_cut = 150.0  # Minimum pT of the Z boson [GeV]

# ── Build the card content from the variables above ───────────────────────
run_card_lines = [
    "#*********************************************************************\n",
    "# MadGraph5_aMC@NLO Run Card for Boosted Z -> b b~ + jet (LO)        *\n",
    "# Process: p p > z j, z > b b~                                       *\n",
    "# Energy: sqrt(s) = 13 TeV                                           *\n",
    f"# Boost cut: ptZmin = {ptZ_cut} GeV / ptj = {ptj_cut} GeV                    *\n",
    "#*********************************************************************\n",
    "#\n",
    "# Event Generation Details\n",
    "#\n",
    f" {nevents:<8} = nevents     ! Number of unweighted events requested\n",
    f" {iseed:<8} = iseed       ! Random seed (0 = auto-generate)\n",
    "#\n",
    "# Beam Physics\n",
    "#\n",
    " 6500.0   = ebeam1      ! Beam 1 energy in GeV  (13 TeV CoM = 6500 + 6500)\n",
    " 6500.0   = ebeam2      ! Beam 2 energy in GeV\n",
    " 1        = lpp1        ! Beam 1 type (1 = proton)\n",
    " 1        = lpp2        ! Beam 2 type (1 = proton)\n",
    "#\n",
    "# Kinematic Cuts (Boosted Selection)\n",
    "#\n",
    f" {ptj_cut:<8} = ptj         ! Minimum pT for the recoiling jet [GeV]\n",
    f" {ptZ_cut:<8} = ptZmin      ! Minimum pT for the Z boson [GeV]\n",
    " 0.0      = ptb         ! Minimum pT for b quarks [GeV]\n",
    " 5.0      = etaj        ! Maximum |eta| for jets\n",
    " 5.0      = etab        ! Maximum |eta| for b quarks\n",
    " 0.4      = drjj        ! Minimum Delta R between jets\n",
    " 0.0      = drbb        ! Minimum Delta R between b quarks\n",
    "#\n",
    "# Renormalization & Factorization Scales\n",
    "#\n",
    " -1.0     = dynamical_scale_choice ! -1 = default central scale\n",
]
run_card_content = ''.join(run_card_lines)

run_card_path = os.path.join(cards_dir, "zbbj_run_card.dat")
with open(run_card_path, "w") as f:
    f.write(run_card_content)

# ── Print the card to confirm the values were substituted correctly ───────
print(f"=== Run Card written to: {run_card_path} ===\n")
print(run_card_content)


---

> [!IMPORTANT]
> ### Exercise 4: Card Parameter Review & Understanding
> Inspect the card contents printed above and answer the following questions:
> 1. What collision center-of-mass energy ($\sqrt{s}$) is configured by `ebeam1 = 6500` and `ebeam2 = 6500`?
> 2. What does the comma in `generate p p > z j, z > b b~` do?
> 3. Why do we set `ptj = 150` and `ptZmin = 150` in the run card? What physics topology does this cut enforce?

<details>
<summary>Click to show Exercise 4 Reference Solution</summary>

<p>1. <b>Collision CoM Energy</b>: $\sqrt{s} = 6500\text{ GeV} + 6500\text{ GeV} = 13000\text{ GeV} = 13\text{ TeV}$.</p>
<p>2. <b>Decay Syntax</b>: The comma <code>, z &gt; b b~</code> instructs MadGraph to force the intermediate $Z$ boson to decay into $b\bar{b}$, rather than letting it remain stable.</p>
<p>3. <b>Boost Cut Purpose</b>: <code>ptj = 150.0</code> and <code>ptZmin = 150.0</code> enforce a hard transverse momentum recoil ($p_T &gt; 150\text{ GeV}$), creating a boosted $Z$ topology where both $b$ quarks are collimated.</p>

</details>

## Step 5a: Test Run — 10 Events & Execution Log Reading

Before running the full production, we do a fast **10-event test** to confirm the setup compiles and runs correctly.
We reuse the cards written in Step 4 and override `nevents` to `10` at launch time — no need to rewrite the card.

### What to look for in the log
1. Feynman diagram generation and matrix-element compilation.
2. Phase space integration and cross-section ($\sigma$ in pb).
3. Event generation and compression into `.lhe.gz`.

In [ ]:
# ── Test run: 10 events ───────────────────────────────────────────────────────
# We read the proc card written in Step 4 and append launch-time overrides.
# The output directory is renamed to Zbbj_test to keep it separate from the
# production output (Zbbj_LO) written in Step 5b.

test_nevents = 10   # <── change this if you want more/fewer test events

# Read the proc card from Step 4 (contains import / generate / output / launch)
with open(proc_card_path) as f:
    proc_card_content = f.read()

# Replace the output directory name so test events don't land in Zbbj_LO
batch_test_script = proc_card_content.replace("Zbbj_LO", "Zbbj_test") + (
    f"set nevents {test_nevents}\n"   # override -1 placeholder in run card
    f"set run_card {run_card_path}\n" # use the run card from Step 4
    f"done\n"
)

batch_test_path = "run_zbbj_test.mg5"
with open(batch_test_path, "w") as f:
    f.write(batch_test_script)

print(f"Executing {test_nevents}-event test run — streaming MadGraph log below...")
!{mg5_exe} {batch_test_path}

test_lhe = "Zbbj_test/Events/run_01/unweighted_events.lhe.gz"
assert os.path.exists(test_lhe), f"ERROR: Test LHE file missing at {test_lhe}"
print(f"\nSUCCESS: Test run completed. LHE file: {test_lhe}")


> [!IMPORTANT]
> ### Self-Reflection & Log Checkpoint 5.1
> 1. Read the stdout execution log above. What cross section ($\sigma$) was calculated by MadGraph for $pp \to Z+j, Z \to b\bar{b}$ with $p_T^j > 150\text{ GeV}$?
> 2. How many Feynman diagrams were generated for this hard scattering process?

<details>
<summary>Click to show Log Checkpoint 5.1 Reference Solution</summary>

<p>1. <b>Calculated Cross Section</b>: Found in the launch phase space integration step in the log: <code>Cross-section : X.XX +- Y.YY (pb)</code>.</p>
<p>2. <b>Feynman Diagrams</b>: Output near the top of process generation: <code>X diagrams generated</code>.</p>

</details>

## Step 5b: Production Run

The test run passed. Now launch the full production using the same cards.
Set `prod_nevents` below to control how many events to generate.

The output `Zbbj_LO/Events/run_01/unweighted_events.lhe.gz` will be parsed in Notebook 02.

In [ ]:
# ── Production run ────────────────────────────────────────────────────────────
# Reads the same proc card from Step 4; only nevents is overridden at launch time.

prod_nevents = 1000  # <── set the number of production events here

# Read the proc card from Step 4 (contains import / generate / output / launch)
with open(proc_card_path) as f:
    proc_card_content = f.read()

# Append run-time overrides (proc card already targets Zbbj_LO)
batch_prod_script = proc_card_content + (
    f"set nevents {prod_nevents}\n"   # override -1 placeholder in run card
    f"set run_card {run_card_path}\n" # use the run card from Step 4
    f"done\n"
)

batch_prod_path = "run_zbbj_prod.mg5"
with open(batch_prod_path, "w") as f:
    f.write(batch_prod_script)

print(f"Executing {prod_nevents}-event production run (~1-2 mins)...")
!{mg5_exe} {batch_prod_path}

lhe_path = "Zbbj_LO/Events/run_01/unweighted_events.lhe.gz"
assert os.path.exists(lhe_path), f"ERROR: Missing production LHE at {lhe_path}"
file_size_kb = os.path.getsize(lhe_path) / 1024

print(f"\n==========================================")
print(f"SUCCESS: {prod_nevents}-event production run completed!")
print(f"LHE file : {lhe_path}")
print(f"File size: {file_size_kb:.2f} KB")
print(f"==========================================")


## Step 6: Les Houches Event (LHE) Structure Inspection

An LHE file stores event records in XML format.

### Key Record Columns:
- `PID`: $+5 = b$, $-5 = \bar{b}$, $23 = Z$, $21 = g$, $1..4 = u,d,c,s$.
- `Status`: `-1` (incoming parton), `+1` (outgoing final-state parton), `+2` (intermediate decayed resonance).
- `Mother1, Mother2`: Indices pointing to parent particles.
- `Px, Py, Pz, E, Mass`: Particle four-momentum components in GeV.

In [ ]:
import gzip

lhe_path = "Zbbj_test/Events/run_01/unweighted_events.lhe.gz"

print(f"Inspecting test LHE file: {lhe_path}")
with gzip.open(lhe_path, "rt") as f:
    lines = f.readlines()

print(f"Total lines in LHE file: {len(lines)}")

event_lines = []
recording = False
for line in lines:
    if "<event>" in line:
        recording = True
    if recording:
        event_lines.append(line)
    if "</event>" in line:
        break

print("\n--- Sample First LHE Event Block ---")
print("".join(event_lines))


### Detailed Guide: Understanding the LHE Event Block

Here is the exact line-by-line breakdown of the sample `<event>` block printed above:

```text
<event>
 6      1 +1.7098000e+03 1.34422000e+02 7.54677100e-03 1.22103800e-01
        2 -1    0    0  503    0 -0.0000000000e+00 +0.0000000000e+00 +1.3056460859e+03 1.3056460859e+03 0.0000000000e+00 0.0000e+00 -1.0000e+00
       21 -1    0    0  502  503 +0.0000000000e+00 -0.0000000000e+00 -1.0425878548e+01 1.0425878548e+01 0.0000000000e+00 0.0000e+00 1.0000e+00
       23  2    1    2    0    0 -1.7288479548e+01 -9.7330351250e+01 +7.6215368438e+02 7.7391699886e+02 9.1088932974e+01 0.0000e+00 9.0000e+00
        5  1    3    3  501    0 +1.0339326251e+01 -5.6052474293e+00 +3.6958145973e+02 3.6979841244e+02 4.7000000000e+00 0.0000e+00 -1.0000e+00
       -5  1    3    3    0  501 -2.7627805799e+01 -9.1725103821e+01 +3.9257222465e+02 4.0411858642e+02 4.7000000000e+00 0.0000e+00 1.0000e+00
        2  1    1    2  502    0 +1.7288479548e+01 +9.7330351250e+01 +5.3306652301e+02 5.4215496562e+02 0.0000000000e+00 0.0000e+00 -1.0000e+00
<mgrwt>
...
</mgrwt>
</event>
```

#### 1. Header Line (`<event>` line 1)
```text
 6      1 +1.7098000e+03 1.34422000e+02 7.54677100e-03 1.22103800e-01
```
- `NUP = 6`: Number of particle entries in this event.
- `IDPRUP = 1`: Process identification index.
- `XWGTUP = +1.7098000e+03`: Monte Carlo weight $w_i$ of this event (in pb).
- `SCALUP = 134.42`: Factorization/Renormalization scale $\mu_F = \mu_R \approx 134.4\text{ GeV}$.
- `AQED = 0.007547`: Electroweak coupling $\alpha_{\text{QED}}$ at scale $\mu$.
- `AQCD = 0.122104`: Strong coupling constant $\alpha_s$ at scale $\mu$.

#### 2. Particle Entry Columns (13 Columns per Particle Line)
Each particle row contains 13 standard Les Houches fields:
`IDUP  ISTUP  MOTHUP(1) MOTHUP(2)  ICOLUP(1) ICOLUP(2)  Px  Py  Pz  E  Mass  VTIMUP  SPINUP`

- **Line 1 (`PID = 2`, `status = -1`)**: Incoming initial-state $u$ quark ($p_z = +1305.6\text{ GeV}$) from Beam 1. Color tag `503`.
- **Line 2 (`PID = 21`, `status = -1`)**: Incoming initial-state gluon ($p_z = -10.4\text{ GeV}$) from Beam 2. Color tags `502` / `503`.
- **Line 3 (`PID = 23`, `status = 2`)**: Intermediate $Z$ boson ($m_Z = 91.09\text{ GeV}$, $p_T^Z \approx 98.8\text{ GeV}$). Mother indices: `1 2` ($u g \to Z j$).
- **Line 4 (`PID = 5`, `status = 1`)**: Outgoing final-state $b$ quark ($p_T \approx 11.8\text{ GeV}$, $E \approx 369.8\text{ GeV}$, $m = 4.7\text{ GeV}$). Mother index: `3 3` (decayed from $Z$). Color tag `501`.
- **Line 5 (`PID = -5`, `status = 1`)**: Outgoing final-state $\bar{b}$ anti-quark ($p_T \approx 95.8\text{ GeV}$, $E \approx 404.1\text{ GeV}$, $m = 4.7\text{ GeV}$). Mother index: `3 3` (decayed from $Z$). Anti-color tag `501`.
- **Line 6 (`PID = 2`, `status = 1`)**: Outgoing matrix-element recoiling $u$ quark jet ($p_T \approx 98.8\text{ GeV}$, $E \approx 542.2\text{ GeV}$). Mother indices: `1 2`. Color tag `502`.

### Visualizing Feynman Diagrams (`feynman_diagrams.pdf`)

MadGraph automatically draws Feynman diagrams for all matrix elements and saves them as PostScript (`.ps`) files inside `Zbbj_test/SubProcesses/P1_<process_name>/matrix*.ps`.

#### Direct Viewing inside Google Colab:
Instead of manually downloading individual `.ps` files, run the cell below to:
1. Merge all PostScript Feynman diagrams into a **single consolidated PDF file**: `/content/feynman_diagrams.pdf`.
2. Automatically render and display all Feynman diagrams **inline directly inside your Google Colab notebook cell output**!

In [ ]:
# ── Merge all Feynman Diagrams into a Single PDF & Display Inline in Colab ───
import os
import glob
import shutil
import subprocess
from IPython.display import display, Image

# 1. Automatically install Ghostscript & Poppler tools if missing in Google Colab
if not shutil.which("gs") or not shutil.which("pdftoppm"):
    print("Installing Ghostscript & Poppler tools for PDF conversion (~5s)...")
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ghostscript", "poppler-utils"], check=False)

# 2. Collect all PostScript diagram files from the test run
ps_files = sorted(glob.glob("Zbbj_test/SubProcesses/P1_*/matrix*.ps"))

if not ps_files:
    print("Notice: No .ps Feynman diagram files found in Zbbj_test/SubProcesses/")
else:
    print(f"Found {len(ps_files)} Feynman diagram files. Merging into single PDF...")
    merged_pdf = "feynman_diagrams.pdf"
    abs_pdf_path = os.path.abspath(merged_pdf)
    
    # 3. Merge all .ps files into one PDF using Ghostscript
    try:
        cmd = ["gs", "-q", "-dNOPAUSE", "-dBATCH", "-sDEVICE=pdfwrite", f"-sOutputFile={merged_pdf}"] + ps_files
        subprocess.run(cmd, check=True)
        print(f"SUCCESS: Created merged PDF -> {abs_pdf_path}")

        # 4. Convert PDF pages to PNG and display inline inside Colab
        subprocess.run(["pdftoppm", "-png", "-r", "150", merged_pdf, "feynman_page"], check=True)
        png_pages = sorted(glob.glob("feynman_page-*.png"))
        
        print(f"\n--- Inline Feynman Diagram Visualizations ({len(png_pages)} pages) ---")
        for img in png_pages:
            display(Image(filename=img))

    except Exception as e:
        print(f"Notice: Could not generate inline PDF preview: {e}")


> [!IMPORTANT]
> ### Exercise 6: LHE Particle Record Identification
> In the output of the first `<event>` block above, answer the following questions:
> 1. What are the PIDs of the incoming initial-state partons ($status = -1$)?
> 2. Find the intermediate $Z$ boson ($PID = 23$). What is its status code?
> 3. Find the final-state bottom quarks ($PID = 5$ and $PID = -5$). What are their status codes?
> 4. Where is the Monte Carlo event weight $w_i$ located in the `<event>` header line?

<details>
<summary>Click to show Exercise 6 Reference Solution</summary>

<p>1. <b>Incoming Partons</b>: Quarks ($PID = 1..4, -1..-4$) or gluons ($PID = 21$) with $status = -1$.</p>
<p>2. <b>Intermediate $Z$ Boson</b>: $PID = 23$ with $status = 2$ (decayed resonance).</p>
<p>3. <b>Final-State Bottom Quarks</b>: $PID = 5$ ($b$) and $PID = -5$ ($\bar{b}$) with $status = 1$.</p>
<p>4. <b>Event Weight</b>: Located as the 3rd numerical value in the first header line of the <code>&lt;event&gt;</code> block.</p>

</details>